In [8]:
import cv2
import numpy as np

# Leer imagen
imagen = cv2.imread("img/IMG_20191209_100620.jpg")
gray = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# =========================
# 1. DETECTAR DADOS (como código 2)
# =========================

blur = cv2.GaussianBlur(gray, (5,5), 0)

_, bin_img = cv2.threshold(blur, 0, 255,
                           cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = np.ones((3,3), np.uint8)
bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_CLOSE, kernel)
bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, kernel)

# Componentes conectados (dados)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(bin_img, 8)

resultado = imagen.copy()
contador_dados = 0

for i in range(1, num_labels):

    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]

    # FILTRAR DADOS
    if 3000 < area < 30000:

        contador_dados += 1

        # =========================
        # 2. CREAR MÁSCARA DEL DADO
        # =========================
        mask = (labels == i).astype(np.uint8) * 255

        roi_gray = gray[y:y+h, x:x+w]
        roi_mask = mask[y:y+h, x:x+w]

        # Aplicar máscara (clave para eliminar fondo)
        roi_masked = roi_gray.copy()
        roi_masked[roi_mask == 0] = 255

        # =========================
        # 3. DETECTAR PUNTOS (como código 1)
        # =========================

        _, puntos_bin = cv2.threshold(roi_masked, 0, 255,
                                     cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        puntos_bin = cv2.morphologyEx(puntos_bin, cv2.MORPH_OPEN,
                                     np.ones((2,2), np.uint8))

        n_labels, _, stats_p, _ = cv2.connectedComponentsWithStats(puntos_bin, 8)

        puntos = 0

        for j in range(1, n_labels):
            area_p = stats_p[j, cv2.CC_STAT_AREA]

            # FILTRO DE PUNTOS
            if 30 < area_p < 700:
                puntos += 1

        # =========================
        # 4. DIBUJAR RESULTADO
        # =========================

        cx = x + w // 2
        cy = y + h // 2

        cv2.rectangle(resultado, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(resultado, str(puntos),
                    (cx-10, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

print("Total de dados:", contador_dados)

# Mostrar
alto, ancho = resultado.shape[:2]
escala = 800 / ancho
mostrar = cv2.resize(resultado, None, fx=escala, fy=escala)

cv2.imshow("Resultado", mostrar)
cv2.waitKey(0)
cv2.destroyAllWindows()

Total de dados: 4
